# IduEdu paper v2 — figures

Все фигуры собираются из `results/*.csv` (см. `benchmarks/`); оформление повторяет фигуры v1
(`dev/iduedu_paper_images.ipynb`). Если данных ещё нет — ячейка печатает предупреждение.
PNG сохраняются рядом; статья ссылается на них из `paper/figures/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

RESULTS = Path("../results")
FIGURES = Path(".")
DPI = 300

# --- v1 palette ---
C = {
    "iduedu": "#00A8FF",
    "iduedu_light": "#5AC8FA",
    "osmnx": "#AF52DE",
    "osmnx_light": "#D66BFF",
    "pyrosm": "#FF9F0A",
    "networkit": "#AF52DE",
    "igraph": "#FF9F0A",
    "networkx": "#FF375F",
    "green": "#32D74B",
    "gray": "#888888",
}
IDUEDU_CUTOFF_STYLE = {
    "5":  dict(color="#9EC5FE", linestyle=":",  linewidth=1.2),
    "15": dict(color="#6EA8FE", linestyle="--", linewidth=1.3),
    "30": dict(color="#3D8BFD", linestyle="-.", linewidth=1.4),
    "60": dict(color="#0D6EFD", linestyle="-",  linewidth=1.5),
    "inf": dict(color="#0A58CA", linestyle="-", linewidth=2.6),
}
STROKE = [pe.Stroke(linewidth=2.0, foreground="white"), pe.Normal()]


def style_ax(ax, ygrid_only=True):
    ax.set_facecolor("white")
    for spine in ax.spines.values():
        spine.set_visible(False)
    if ygrid_only:
        ax.yaxis.grid(True, color="black", alpha=0.10, linewidth=0.8, zorder=1)
    else:
        ax.grid(True, which="both", color="black", alpha=0.10, linewidth=0.8)
    ax.tick_params(axis="both", colors="black", labelsize=10)
    ax.tick_params(which="minor", length=0)


def format_edges(n) -> str:
    n = int(n)
    if n >= 1_000_000:
        return f"{n / 1_000_000:.2f} M"
    if n >= 1_000:
        return f"{n / 1_000:.1f} k"
    return str(n)


def format_mb(mb) -> str:
    if pd.isna(mb):
        return "n/a"
    mb = float(mb)
    if mb >= 1024:
        return f"{mb / 1024:.1f} GB"
    return f"{mb:.0f} MB"


def load(name: str):
    path = RESULTS / name
    if not path.exists():
        print(f"[skip] {name} not found - run the corresponding benchmark first")
        return None
    return pd.read_csv(path)


def save(fig, name: str):
    fig.patch.set_facecolor("white")
    fig.tight_layout()
    fig.savefig(FIGURES / name, dpi=DPI, bbox_inches="tight")
    print(f"saved {name}")

## B1 — Graph construction (walk/drive)

In [ ]:
# Grouped bars per city: x-tick labels are edge counts, city names below (v1 layout).
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

# Editable: readable tick values (seconds) for the log y-axis; filtered to data range.
BUILD_TICKS = [5, 10, 25, 50, 100, 150, 200, 300, 450, 600, 900]


def fmt_seconds(y, _):
    if y >= 1:
        return f"{y:.0f} s"
    return f"{y * 1000:.0f} ms"


df = load("build_benchmark.csv")
if df is not None:
    df = df[df.area != "smoke"].copy()
    df["simplify"] = df["simplify"].replace({"NA": "False"}).astype(str)

    def method_name(row):
        lib = row.library
        if lib == "iduedu":
            return "IduEdu (simp)" if row.simplify == "True" else "IduEdu"
        if lib == "osmnx":
            return "OSMnx (simp)" if row.simplify == "True" else "OSMnx"
        return "Pyrosm"

    df["method"] = df.apply(method_name, axis=1)
    METHOD_ORDER = ["IduEdu", "IduEdu (simp)", "OSMnx", "OSMnx (simp)", "Pyrosm"]
    METHOD_COLORS = {
        "IduEdu": C["iduedu"], "IduEdu (simp)": C["iduedu_light"],
        "OSMnx": C["osmnx"], "OSMnx (simp)": C["osmnx_light"], "Pyrosm": C["pyrosm"],
    }

    for network in df.network.unique():
        sub = df[df.network == network]
        agg = (sub.groupby(["area", "method"], as_index=False)
                  .agg(time_mean=("time_sec", "mean"), time_std=("time_sec", "std"),
                       n_edges=("n_edges", "max")))
        areas = agg.groupby("area")["n_edges"].max().sort_values().index.to_list()

        BAR_W, GAP_CITY = 0.22, 0.30
        xs, hs, errs, cols, etxt = [], [], [], [], []
        city_groups = []
        cursor = 0.0
        for area in areas:
            g = agg[agg.area == area].set_index("method").reindex(METHOD_ORDER).dropna(subset=["time_mean"])
            x_left = cursor
            for m, row in g.iterrows():
                xs.append(cursor); hs.append(row.time_mean)
                errs.append(0.0 if np.isnan(row.time_std) else row.time_std)
                cols.append(METHOD_COLORS[m]); etxt.append(row.n_edges)
                cursor += BAR_W
            city_groups.append((area, x_left, cursor, (x_left + cursor) / 2))
            cursor += GAP_CITY

        fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)
        ax.bar(xs, hs, width=BAR_W, align="edge", color=cols,
               edgecolor="white", linewidth=0.8, alpha=0.95, zorder=3)
        ax.errorbar([x + BAR_W / 2 for x in xs], hs, yerr=errs, fmt="none",
                    ecolor="black", elinewidth=0.9, capsize=2, alpha=0.55, zorder=4)
        style_ax(ax)

        # Log scale with human-readable fixed ticks (seconds, then minutes past 60 s).
        ax.set_yscale("log")
        ymax = max(hs)
        ticks = [t for t in BUILD_TICKS if t <= ymax * 1.25]
        ax.yaxis.set_major_locator(FixedLocator(ticks))
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_seconds))
        ax.yaxis.set_minor_locator(NullLocator())
        ax.set_ylim(top=ymax * 1.25)

        ax.set_ylabel("Build time", fontsize=13)
        ax.set_xticks([x + BAR_W / 2 for x in xs])
        ax.set_xticklabels([format_edges(n) for n in etxt], rotation=40, ha="right", fontsize=9)
        ax.set_xlabel(f"Edges count in resulting {network} graph", fontsize=12, labelpad=42)
        for area, x_l, x_r, x_c in city_groups:
            ax.text(x_c, -0.17, area, ha="center", va="top", transform=ax.get_xaxis_transform(),
                    fontsize=12, fontweight="bold", path_effects=STROKE, clip_on=False)
        present = [m for m in METHOD_ORDER if m in agg.method.unique()]
        ax.legend(handles=[Patch(facecolor=METHOD_COLORS[m], label=m) for m in present],
                  loc="upper left", frameon=True, fontsize=9)
        save(fig, f"build_benchmark_{network}.png")

## B1-абляция — simplify on/off

In [ ]:
# IduEdu simplify ablation: walk and drive have different simplify trade-offs.
df = load("build_benchmark.csv")
if df is not None:
    sub = df[(df.library == "iduedu") & (df.area != "smoke") & (df.network.isin(["walk", "drive"]))].copy()
    if not sub.empty:
        sub["simplify"] = sub["simplify"].astype(str)
        sub["representation_size_mb"] = pd.to_numeric(sub.get("representation_size_mb", sub.get("graph_memory_mb", np.nan)), errors="coerce")
        agg = (sub.groupby(["network", "area", "simplify"], as_index=False)
                  .agg(time_mean=("time_sec", "mean"),
                       n_edges=("n_edges", "max"),
                       representation_size_mb=("representation_size_mb", "median")))
        areas = agg.groupby("area")["n_edges"].max().sort_values().index.to_list()
        COLORS = {"False": C["iduedu"], "True": C["iduedu_light"]}
        ABLATION_TICKS = [1, 2, 3, 5, 7.5, 10, 15, 20, 30, 45, 60, 90, 120, 180, 300]

        fig, axes = plt.subplots(2, 1, figsize=(13, 8), dpi=DPI, sharey=False)
        for ax, network in zip(axes.ravel(), ["walk", "drive"]):
            net = agg[agg.network == network].copy()
            if net.empty:
                ax.set_visible(False)
                continue

            BAR_W, GAP = 0.22, 0.28
            xs, hs, cols, etxt = [], [], [], []
            city_groups = []
            cursor = 0.0
            for area in areas:
                g = net[net.area == area].set_index("simplify").reindex(["False", "True"]).dropna(subset=["time_mean"])
                if g.empty:
                    continue
                x_left = cursor
                for s, row in g.iterrows():
                    xs.append(cursor); hs.append(row.time_mean); cols.append(COLORS[s]); etxt.append(row.n_edges)
                    cursor += BAR_W
                city_groups.append((area, x_left, cursor, (x_left + cursor) / 2))
                cursor += GAP

            ax.bar(xs, hs, width=BAR_W, align="edge", color=cols,
                   edgecolor="white", linewidth=0.8, alpha=0.95, zorder=3)
            style_ax(ax)
            ax.set_title(network.capitalize(), fontsize=13, fontweight="bold")
            ax.set_yscale("log")
            ymax = max(hs)
            ymin = max(min(hs) * 0.75, 0.5)
            ticks = [t for t in ABLATION_TICKS if ymin <= t <= ymax * 1.55]
            ax.yaxis.set_major_locator(FixedLocator(ticks))
            ax.yaxis.set_major_formatter(FuncFormatter(fmt_seconds))
            ax.yaxis.set_minor_locator(NullLocator())
            ax.set_ylim(ymin, ymax * 1.55)
            ax.set_ylabel("Build time", fontsize=13)
            ax.set_xticks([x + BAR_W / 2 for x in xs])
            ax.set_xticklabels([format_edges(n) for n in etxt], rotation=40, ha="right", fontsize=9)
            ax.set_xlabel("Edges count in resulting graph", fontsize=11, labelpad=46)

            piv = net.pivot(index="area", columns="simplify", values=["time_mean", "n_edges", "representation_size_mb"])
            for area, x_l, x_r, x_c in city_groups:
                ax.text(x_c, -0.26, area, ha="center", va="top", transform=ax.get_xaxis_transform(),
                        fontsize=9.5, fontweight="bold", path_effects=STROKE, clip_on=False)
                try:
                    t_raw = float(piv.loc[area, ("time_mean", "False")])
                    t_smp = float(piv.loc[area, ("time_mean", "True")])
                    e_raw = int(piv.loc[area, ("n_edges", "False")])
                    e_smp = int(piv.loc[area, ("n_edges", "True")])
                    m_raw = float(piv.loc[area, ("representation_size_mb", "False")])
                    m_smp = float(piv.loc[area, ("representation_size_mb", "True")])
                except (KeyError, ValueError, TypeError):
                    continue

                mem_line = "size n/a"
                if np.isfinite(m_raw) and np.isfinite(m_smp) and m_raw > 0:
                    mem_line = f"size {(m_smp / m_raw - 1) * 100:+.0f}%"
                txt = f"time x{t_smp / t_raw:.2f}\nedges {(e_smp / e_raw - 1) * 100:+.0f}%\n{mem_line}"
                ax.text(x_c, max(t_raw, t_smp) * 1.10, txt, ha="center", va="bottom",
                        fontsize=8.8, linespacing=1.05, path_effects=STROKE, zorder=12)

        axes[0].legend(handles=[Patch(facecolor=COLORS["False"], label="IduEdu"),
                                Patch(facecolor=COLORS["True"], label="IduEdu (simp)")],
                       loc="upper left", frameon=True, fontsize=9)
        fig.subplots_adjust(bottom=0.18, hspace=0.78)
        save(fig, "ablation_simplify.png")

## B2 — Multimodal pipeline: stacked stages

In [ ]:
# Multimodal: HORIZONTAL stacked bars (v1 layout) - PT/Walk/Join, edge labels in segments.
df = load("intermodal_benchmark.csv")
if df is not None:
    df = df[df.area != "smoke"]
    if not df.empty:
        agg = (df.groupby("area", as_index=False)
                 .agg(t_walk=("time_walk_sec", "median"), t_pt=("time_pt_sec", "median"),
                      t_join=("time_join_sec", "median"), t_total=("time_total_sec", "median"),
                      e_walk=("n_edges_walk", "median"), e_pt=("n_edges_pt", "median"),
                      e_inter=("n_edges_intermodal", "median")))
        agg = agg.sort_values("t_total").reset_index(drop=True)
        y = np.arange(len(agg))
        pt = agg.t_pt.to_numpy()
        walk = agg.t_walk.to_numpy()
        join = agg.t_join.clip(lower=0).to_numpy()
        total = pt + walk + join
        COL = {"PT": "#00A8FF", "Walk": "#FF9F0A", "Join": "#AF52DE"}

        fig, ax = plt.subplots(figsize=(12, 4), dpi=DPI)
        ax.set_facecolor("white")
        ax.barh(y, pt, color=COL["PT"], edgecolor="white", linewidth=0.8, alpha=0.95, label="PT")
        ax.barh(y, walk, left=pt, color=COL["Walk"], edgecolor="white", linewidth=0.8, alpha=0.95, label="Walk")
        ax.barh(y, join, left=pt + walk, color=COL["Join"], edgecolor="white", linewidth=0.8, alpha=0.95,
                label="Multi-modal (join)")
        ax.set_yticks(y)
        ax.set_yticklabels(agg.area, fontsize=12)
        ax.invert_yaxis()
        ax.set_xlabel("Build time (s)", fontsize=12)
        ax.xaxis.grid(True, color="black", alpha=0.10, linewidth=0.8, zorder=1)
        for spine in ax.spines.values():
            spine.set_visible(False)
        ax.tick_params(axis="both", colors="black")
        ax.set_xlim(0, total.max() * 1.18)
        for i in range(len(agg)):
            ax.text(pt[i] / 2, y[i], format_edges(agg.e_pt[i]), ha="center", va="center",
                    fontsize=10, fontweight="bold", path_effects=STROKE)
            ax.text(pt[i] + walk[i] / 2, y[i], format_edges(agg.e_walk[i]), ha="center", va="center",
                    fontsize=10, fontweight="bold", path_effects=STROKE)
            ax.text(total[i] + total.max() * 0.015, y[i], f"= {format_edges(agg.e_inter[i])} edges",
                    ha="left", va="center", fontsize=10, fontweight="bold", path_effects=STROKE,
                    clip_on=False)
        leg = ax.legend(loc="upper right", frameon=True, fontsize=11, title="Component", title_fontsize=11)
        leg.get_frame().set_facecolor("white")
        leg.get_frame().set_edgecolor("black")
        save(fig, "intermodal_stacked_time_by_city.png")

## B3 — OD matrices: rect / square

In [ ]:
from matplotlib.ticker import FuncFormatter, LogLocator, NullFormatter

XTICKS2 = [128, 256, 512, 1024, 2048, 4096, 8192]
XTICKS1 = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072]


def fmt_seconds(y, _):
    if y >= 3600:
        return f"{y / 3600:.0f} h"
    if y >= 60:
        return f"{y / 60:.0f} min"
    if y >= 1:
        return f"{y:.0f} s"
    return f"{y * 1000:.0f} ms"


df = load("od_benchmark.csv")
if df is not None:
    for mode in ["rect", "square"]:
        sub = df[(df["mode"] == mode) & (df.n_sources >= 128)].copy()
        if sub.empty:
            print(f"[skip] no {mode} data yet")
            continue
        # pandas 3: astype(str) keeps NaN as null (not "nan") - resolve via isna()
        sub["thr"] = np.where(sub.threshold_min.isna(), "inf",
                              sub.threshold_min.fillna(0).astype(int).astype(str))
        # competitors: end-to-end (snap + convert + od); iduedu: od time (includes snap by design)
        sub["t"] = np.where(sub.library == "iduedu", sub.time_od_sec, sub.time_total_sec)
        agg = (sub.groupby(["library", "thr", "n_sources"], as_index=False)
                  .agg(time_med=("t", "median")))

        OTHER = {"networkit": ("NetworKit", C["networkit"]),
                 "igraph": ("igraph", C["igraph"]),
                 "networkx": ("NetworkX", C["networkx"])}

        fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)
        ax.set_xscale("log")
        ax.set_yscale("log")
        for (lib, thr), g in agg.groupby(["library", "thr"]):
            g = g.sort_values("n_sources")
            if lib == "iduedu":
                stl = IDUEDU_CUTOFF_STYLE.get(thr, IDUEDU_CUTOFF_STYLE["inf"])
                label = "IduEdu" if thr == "inf" else f"IduEdu (τ={thr})"
            else:
                name, color = OTHER[lib]
                stl = dict(color=color, linestyle="-", linewidth=2.6)
                label = name
            ax.plot(g.n_sources, g.time_med, label=label, marker="o", markersize=3.5,
                    markerfacecolor="white", markeredgecolor=stl["color"], markeredgewidth=0.8,
                    zorder=4, **stl)
        style_ax(ax, ygrid_only=False)
        ax.set_xlabel("Number of sources |O|", fontsize=11)
        ax.set_ylabel("Total time", fontsize=11)

        XTICKS = XTICKS1 if mode == "rect" else XTICKS2
        ax.set_xticks(XTICKS)
        ax.set_xticklabels([f"{x:,}".replace(",", " ") for x in XTICKS])
        ax.xaxis.set_minor_formatter(NullFormatter())

        # Human-readable time on the log y-axis: label every decade (ms / s / min)
        # plus the 2x/5x minors so the magnitudes are easy to read off.
        ax.yaxis.set_major_locator(LogLocator(base=10))
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs=(2.0, 5.0)))
        
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_seconds))
        ax.yaxis.set_minor_formatter(FuncFormatter(fmt_seconds))
        
        ax.tick_params(axis="y", which="minor", labelsize=7, colors="0.35")
        ax.tick_params(axis="y", which="major", labelsize=10)

        leg = ax.legend(loc="upper left", frameon=True, fontsize=9,
                        title="Library / cutoff", title_fontsize=10,
                        borderpad=0.8, labelspacing=0.55, handlelength=3.0)
        leg.get_frame().set_facecolor("white")
        leg.get_frame().set_edgecolor("black")
        leg.get_frame().set_alpha(0.95)
        save(fig, f"od_time_{mode}.png")

## B3b — таблица ускорения от cutoff (для `tab:od_cutoff_square`)

In [ ]:
# Cutoff speed-up table for tab:od_cutoff_square.
df = load("od_benchmark.csv")
if df is not None:
    sub = df[(df["mode"] == "square") & (df.library == "iduedu")]
    if not sub.empty:
        med = sub.groupby(["threshold_min", "n_sources"], dropna=False)["time_od_sec"].median().reset_index()
        full = med[med.threshold_min.isna()].set_index("n_sources")["time_od_sec"]
        cut = med[med.threshold_min.notna()].copy()
        cut["speedup"] = cut.apply(lambda r: full.get(r.n_sources, np.nan) / r.time_od_sec, axis=1)
        cut["threshold_min"] = cut["threshold_min"].astype(int)
        table = cut.pivot(index="threshold_min", columns="n_sources", values="speedup").round(2)
        print(table.to_string())
        print()
        print(table.to_latex(float_format="%.2f"))

## B1 (память) — размер представления графа

In [ ]:
# In-memory graph representation size from B1 build benchmark.
df = load("build_benchmark.csv")
if df is not None:
    df = df[df.area != "smoke"].copy()
    if "representation_size_mb" not in df.columns:
        df["representation_size_mb"] = np.nan
    df["representation_size_mb"] = pd.to_numeric(df["representation_size_mb"], errors="coerce")
    if "graph_memory_mb" in df.columns:
        fallback = pd.to_numeric(df["graph_memory_mb"], errors="coerce")
        df["representation_size_mb"] = df["representation_size_mb"].fillna(fallback)
    df["simplify"] = df["simplify"].replace({"NA": "False"}).astype(str)

    def memory_method(row):
        if row.library == "iduedu" and row.simplify == "True":
            return "IduEdu (simp)"
        if row.library == "osmnx" and row.simplify == "True":
            return "OSMnx (simp)"
        if row.library == "pyrosm":
            return "Pyrosm"
        return None

    df["method"] = df.apply(memory_method, axis=1)
    sub = df.dropna(subset=["method", "representation_size_mb"])
    if sub.method.nunique() < 2:
        print("[skip] representation_size_mb is incomplete - rerun bench_build.py")
    else:
        METHOD_ORDER = ["IduEdu (simp)", "OSMnx (simp)", "Pyrosm"]
        METHOD_COLORS = {
            "IduEdu (simp)": C["iduedu_light"],
            "OSMnx (simp)": C["osmnx_light"],
            "Pyrosm": C["pyrosm"],
        }
        agg = (sub.groupby(["network", "area", "method"], as_index=False)
                  .agg(size_mb=("representation_size_mb", "median"), n_edges=("n_edges", "median")))
        areas = agg.groupby("area")["n_edges"].max().sort_values().index.to_list()

        fig, axes = plt.subplots(2, 1, figsize=(13, 8), dpi=DPI, sharey=False)
        for ax, network in zip(axes.ravel(), ["walk", "drive"]):
            net = agg[agg.network == network].copy()
            if net.empty:
                ax.set_visible(False)
                continue
            BAR_W, GAP_CITY = 0.24, 0.34
            xs, hs, cols = [], [], []
            city_groups = []
            cursor = 0.0
            for area in areas:
                g = net[net.area == area].set_index("method").reindex(METHOD_ORDER).dropna(subset=["size_mb"])
                if g.empty:
                    continue
                x_left = cursor
                for method, row in g.iterrows():
                    xs.append(cursor)
                    hs.append(row.size_mb)
                    cols.append(METHOD_COLORS[method])
                    cursor += BAR_W
                city_groups.append((area, x_left, cursor, (x_left + cursor) / 2))
                cursor += GAP_CITY

            ax.bar(xs, hs, width=BAR_W, align="edge", color=cols,
                   edgecolor="white", linewidth=0.8, alpha=0.95, zorder=3)
            style_ax(ax)
            ax.set_title(network.capitalize(), fontsize=13, fontweight="bold")
            ax.set_yscale("log")
            ax.set_ylabel("Representation size", fontsize=12)
            ax.set_xticks([])
            ymax = max(hs)
            ax.set_ylim(max(min(hs) * 0.75, 1), ymax * 1.55)
            for x, h in zip(xs, hs):
                ax.text(x + BAR_W / 2, h * 1.06, format_mb(h), ha="center", va="bottom",
                        fontsize=8.5, path_effects=STROKE)
            for area, x_l, x_r, x_c in city_groups:
                ax.text(x_c, -0.12, area, ha="center", va="top", transform=ax.get_xaxis_transform(),
                        fontsize=10.5, fontweight="bold", path_effects=STROKE, clip_on=False)

        present = [m for m in METHOD_ORDER if m in agg.method.unique()]
        axes[0].legend(handles=[Patch(facecolor=METHOD_COLORS[m], label=m) for m in present],
                       loc="upper left", frameon=True, fontsize=9)
        fig.subplots_adjust(bottom=0.10, hspace=0.45)
        save(fig, "memory_benchmark.png")

## README/docs benchmark snapshot — pedestrian graph vs OSMnx

Compact figure for the project README and documentation. It uses the same B1 benchmark data and visual style, and shows pedestrian graph construction for `simplify=False` and `simplify=True` in IduEdu and OSMnx as four colored series.


In [ ]:
# README/docs benchmark snapshot: pedestrian graph construction, IduEdu vs OSMnx.
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

df = load("build_benchmark.csv")
if df is not None:
    sub = df[(df.area != "smoke") & (df.network == "walk") & (df.library.isin(["iduedu", "osmnx"]))].copy()
    sub["simplify"] = sub["simplify"].astype(str)
    sub["representation_size_mb"] = pd.to_numeric(sub["representation_size_mb"], errors="coerce")

    def method_name(row):
        if row.library == "iduedu":
            return "IduEdu (simp)" if row.simplify == "True" else "IduEdu"
        return "OSMnx (simp)" if row.simplify == "True" else "OSMnx"

    sub["method"] = sub.apply(method_name, axis=1)
    METHOD_ORDER = ["IduEdu", "IduEdu (simp)", "OSMnx", "OSMnx (simp)"]
    METHOD_COLORS = {
        "IduEdu": C["iduedu"],
        "IduEdu (simp)": C["iduedu_light"],
        "OSMnx": C["osmnx"],
        "OSMnx (simp)": C["osmnx_light"],
    }
    agg = (sub.groupby(["area", "method"], as_index=False)
              .agg(time_sec=("time_sec", "median"),
                   representation_size_mb=("representation_size_mb", "median"),
                   n_edges=("n_edges", "median")))
    areas = agg.groupby("area")["n_edges"].max().sort_values().index.to_list()

    metrics = [
        ("time_sec", "Build time", "seconds", True, lambda v: f"{v:.0f}s"),
        ("representation_size_mb", "Graph object size", "MB", True, format_mb),
        ("n_edges", "Stored edge rows", "edges", False, format_edges),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(14.4, 5.25), dpi=DPI)
    BAR_W = 0.18
    offsets = np.linspace(-1.5 * BAR_W, 1.5 * BAR_W, len(METHOD_ORDER))
    x = np.arange(len(areas))

    for ax, (metric, title, ylabel, use_log, formatter) in zip(axes, metrics):
        vals_all = []
        for offset, method in zip(offsets, METHOD_ORDER):
            values = []
            for area in areas:
                row = agg[(agg.area == area) & (agg.method == method)]
                values.append(float(row.iloc[0][metric]) if not row.empty else np.nan)
            vals_all.extend([v for v in values if not np.isnan(v)])
            bars = ax.bar(x + offset, values, width=BAR_W, color=METHOD_COLORS[method],
                          edgecolor="white", linewidth=0.8, alpha=0.95, label=method, zorder=3)
            for bar, value in zip(bars, values):
                if np.isnan(value):
                    continue
                if metric == "n_edges":
                    label_y = value + max(values) * 0.018
                    rotation = 90
                    fontsize = 6.7
                else:
                    label_y = value * (1.06 if use_log else 1.014)
                    rotation = 0
                    fontsize = 6.9
                ax.text(bar.get_x() + bar.get_width() / 2, label_y, formatter(value),
                        ha="center", va="bottom", fontsize=fontsize, rotation=rotation,
                        path_effects=STROKE, zorder=4)

        style_ax(ax)
        ax.set_title(title, fontsize=13, fontweight="bold")
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_xticks(x)
        ax.set_xticklabels(areas, rotation=34, ha="right", fontsize=8.7)
        if use_log and vals_all:
            ax.set_yscale("log")
            if metric == "time_sec":
                ticks = [5, 10, 25, 50, 100, 250, 500]
                ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f"{y:.0f}s"))
            else:
                ticks = [50, 100, 250, 500, 1000, 2500, 5000]
                ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: format_mb(y)))
            ticks = [tick for tick in ticks if min(vals_all) * 0.62 <= tick <= max(vals_all) * 1.85]
            ax.yaxis.set_major_locator(FixedLocator(ticks))
            ax.yaxis.set_minor_locator(NullLocator())
            ax.set_ylim(min(vals_all) * 0.70, max(vals_all) * 1.62)
        elif vals_all:
            ax.set_ylim(0, max(vals_all) * 1.34)
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: format_edges(y) if y else "0"))

    handles = [Patch(facecolor=METHOD_COLORS[m], label=m) for m in METHOD_ORDER]
    axes[0].legend(handles=handles, loc="upper left", frameon=True, fontsize=8.6, ncols=1)
    fig.suptitle("Pedestrian graph construction benchmark", fontsize=15, fontweight="bold", y=1.03)
    fig.text(0.5, -0.025, "Median of repeated B1 runs; same AOI per city. Edge rows reflect each library's graph representation.",
             ha="center", fontsize=9.5, color="#444444")
    save(fig, "benchmark_walk_summary.png")
    docs_static = Path("../../docs/_static")
    docs_static.mkdir(parents=True, exist_ok=True)
    fig.savefig(docs_static / "benchmark_walk_summary.png", dpi=DPI, bbox_inches="tight")

